# RetailPulse -- Churn Prediction with XGBoost + SHAP

**Objective:** Predict which customers are likely to churn using XGBoost. Use SHAP values to explain predictions.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.model_selection import train_test_split, StratifiedKFold
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (14, 6), "font.size": 12})
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


## Define Churn

A customer is **churned** if they have not purchased in the last 90 days.

In [2]:
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_rfm.csv"))
CHURN_THRESHOLD = 90
rfm["is_churned"] = (rfm["recency"] > CHURN_THRESHOLD).astype(int)
print(f"Total: {len(rfm):,} | Churned: {rfm['is_churned'].sum():,} ({rfm['is_churned'].mean()*100:.1f}%) | Active: {(~rfm['is_churned'].astype(bool)).sum():,}")


Total: 5,878 | Churned: 2,989 (50.9%) | Active: 2,889


## Feature Preparation

In [3]:
feature_cols = ["recency", "frequency", "monetary", "r_score", "f_score", "m_score", "rfm_score"]
X = rfm[feature_cols].copy()
y = rfm["is_churned"].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train churn rate: {y_train.mean()*100:.1f}% | Test churn rate: {y_test.mean()*100:.1f}%")


Train: 4702 | Test: 1176
Train churn rate: 50.9% | Test churn rate: 50.9%


## XGBoost Model

In [4]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    "max_depth": 5, "learning_rate": 0.1, "objective": "binary:logistic",
    "eval_metric": "logloss", "scale_pos_weight": scale_pos, "seed": 42,
}
model = xgb.train(params, dtrain, num_boost_round=200,
                  evals=[(dtest, "test")], verbose_eval=False)

y_prob = model.predict(dtest)
y_pred = (y_prob > 0.5).astype(int)
print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Active", "Churned"]))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")


CLASSIFICATION REPORT


              precision    recall  f1-score   support

      Active       1.00      1.00      1.00       578
     Churned       1.00      1.00      1.00       598

    accuracy                           1.00      1176
   macro avg       1.00      1.00      1.00      1176
weighted avg       1.00      1.00      1.00      1176

ROC AUC: 1.0000


## Cross-Validation

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []
for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    dtrain_cv = xgb.DMatrix(X.iloc[tr_idx], label=y.iloc[tr_idx])
    dval_cv = xgb.DMatrix(X.iloc[val_idx], label=y.iloc[val_idx])
    m = xgb.train(params, dtrain_cv, num_boost_round=200, verbose_eval=False)
    preds = m.predict(dval_cv)
    auc = roc_auc_score(y.iloc[val_idx], preds)
    cv_aucs.append(auc)
print(f"5-Fold CV ROC AUC: {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f}")
print(f"Individual folds: {[round(s, 4) for s in cv_aucs]}")


5-Fold CV ROC AUC: 1.0000 +/- 0.0000
Individual folds: [1.0, 1.0, 1.0, 1.0, 1.0]


## Evaluation Plots

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
cm = confusion_matrix(y_test, y_pred)
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion Matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(["Active","Churned"]); axes[0].set_yticklabels(["Active","Churned"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i,j]}", ha="center", va="center", fontsize=16,
                     color="white" if cm[i,j] > cm.max()/2 else "black")
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="#e74c3c", linewidth=2, label=f"AUC={roc_auc_score(y_test,y_prob):.3f}")
axes[1].plot([0,1],[0,1],"k--",alpha=0.3); axes[1].set_title("ROC Curve"); axes[1].legend()
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(rec, prec, color="#3498db", linewidth=2)
axes[2].set_title("Precision-Recall"); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
fig.suptitle("XGBoost Churn Model Evaluation", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout(); save_fig(fig, "33_churn_evaluation.png"); plt.show()


Saved: 33_churn_evaluation.png


## SHAP Explainability

In [7]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
fig, axes = plt.subplots(1, 2, figsize=(20, 6))
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, show=False, plot_size=None)
axes[0].set_title("SHAP Beeswarm")
plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False, plot_size=None)
axes[1].set_title("SHAP Importance")
fig.tight_layout(); save_fig(fig, "34_shap_analysis.png"); plt.show()


Saved: 34_shap_analysis.png


In [8]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "shap_importance": np.abs(shap_values).mean(axis=0),
}).sort_values("shap_importance", ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(importance["feature"], importance["shap_importance"], color="#e74c3c")
ax.set_xlabel("Mean |SHAP value|"); ax.set_title("Feature Importance (SHAP)")
ax.invert_yaxis(); fig.tight_layout()
save_fig(fig, "35_feature_importance_comparison.png"); plt.show()


Saved: 35_feature_importance_comparison.png


In [9]:
rfm_churn = rfm.copy()
dall = xgb.DMatrix(rfm[feature_cols])
rfm_churn["churn_probability"] = model.predict(dall)
rfm_churn["churn_risk"] = pd.cut(rfm_churn["churn_probability"], bins=[0, 0.3, 0.7, 1.0],
                                  labels=["Low Risk", "Medium Risk", "High Risk"])
rfm_churn.to_csv(os.path.join(PROCESSED_DIR, "customer_churn.csv"), index=False)
print("Saved: customer_churn.csv")
print(rfm_churn["churn_risk"].value_counts().to_string())
print("\nCHURN PREDICTION COMPLETE")


Saved: customer_churn.csv
churn_risk
High Risk      2987
Low Risk       2888
Medium Risk       3

CHURN PREDICTION COMPLETE
